# Лабораторная работа №1 Задача регрессии

## Задание

1. Примените следующие типы регрессий для датасета представленного по
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link).
- Линейная регрессия
- LASSO регрессия
- Ridge регрессия
- ElasticNet регрессия
- Polynom регрессия (для степени полинома 2)
- Polynom регрессия (для степени полинома 5)
- Polynom регрессия (для степени полинома 10)

Постройте модель для атрибутов `X = {SkinThickness, BMI}` и `Y = {Insulin}`.  
**Укажите `score` (r2_score) моделей. Найдите метрики MAE, MSE, RMSE.**


## Загрузка данных

Датасет доступен по ссылке <https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link>

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [ ]:
LOCAL_PATH = "data/diabetes.csv"
GITHUB_RAW_URL = "https://raw.githubusercontent.com/lufikow/intro-ai/refs/heads/main/data/diabetes.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    df = pd.read_csv(GITHUB_RAW_URL)

In [ ]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

## Регрессия разных видов

In [7]:
pd.options.display.float_format = '{:.4f}'.format

cols = ['SkinThickness', 'BMI', 'Insulin']
# df[cols] = df[cols].replace(0, np.nan)
# df = df.dropna(subset=cols)

X = df[['SkinThickness', 'BMI']].values
y = df['Insulin'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

med = pd.DataFrame(X_train).median()
X_train = pd.DataFrame(X_train).replace(0, med)
X_test = pd.DataFrame(X_test).replace(0, med)

models = {
    'Linear': make_pipeline(StandardScaler(), LinearRegression()),
    'Lasso': make_pipeline(StandardScaler(), Lasso(alpha=1, max_iter=100000)),
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1)),
    'ElasticNet': make_pipeline(StandardScaler(), ElasticNet(alpha=1, l1_ratio=1, max_iter=100000)),
    'Poly deg=2': make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), LinearRegression()),
    'Poly deg=5': make_pipeline(PolynomialFeatures(degree=5), StandardScaler(), LinearRegression()),
    'Poly deg=10': make_pipeline(PolynomialFeatures(degree=10), StandardScaler(), LinearRegression())
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
            'model': name,
            'r2_score': r2_score(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'MSE': mean_squared_error(y_test, y_pred),
            'RMSE': root_mean_squared_error(y_test, y_pred)
        })

result_df = pd.DataFrame(results)
result_df

,model,r2_score,MAE,MSE,RMSE
0,Linear,0.0616,83.9888,11556.5911,107.5016
1,Lasso,0.0623,84.0638,11548.0152,107.4617
2,Ridge,0.0616,83.9925,11556.1706,107.4996
3,ElasticNet,0.0623,84.0638,11548.0152,107.4617
4,Poly deg=2,-0.0632,84.4293,13093.0244,114.4248
5,Poly deg=5,-357.9046,240.4566,4419849.2457,2102.3438
6,Poly deg=10,-126312148.6238,100588.9011,1555512866427.4167,1247202.0151


## Вывод

Среди всех примененных видов линейной регрессии ни один не смог уловить закономерность. Самый высокий полученный коэффициент детерминации - 0.06, что несильно выше банального прогноза с помощью средней арифметической. 
Стоит отметить "убежавшие" оценки регрессий с полиномиальными признаками: даже базовые модели не уловили закономерность, что говорит об ее трудности или возможном отсутствии. В это же время полиномиальные признаки добавили в итак несвязанные значения еще большей запутанности. 

Наибольшая проблема при работе с конкретными тремя взятыми признаками - пропуски в них. Значение "0" в них - содержательная ошибка, так как в реальности ни один из этих факторов не может быть нулевым. Следовательно, пропуски необходимо обработать. Для заполнения недостатков можно использовать три метода, но каждый из них неверен:
1. Удалить строчки с прпоусками. Так поступить нельзя, так как пропусков слишком много - можем потерять важные данные.
2. Заполнить пропуски. Этот метод также оказывается неверным, так как может привести либо к утечке данных, либо к неверному прогнозу. 
3. Оставить все как есть. Нельзя допускать в процесс обучения неподготовленные данные. Trash in - trash out.

Я выбрал заполнение пропусков с избежанием утечки данных, что существенно снизило прогностическую способность, но избавило от аномальных прогнозов.

Проблемой более высокого уровня является сама зависимость между переменными: задание требует использования не всего датасета для регрессии, а лишь конкретных признаков, имеющих слабую корреляцию. Как можно заметить в таблице ниже, все коэффициенты корреляции оказались значимыми, но слабо влияющими. Конкретно говоря, индекс BMI показывает слабую зависимость, а толщина кожи - лишь умеренную. 

In [8]:
from scipy.stats import pearsonr, spearmanr, kendalltau

rows = []
for f in ['SkinThickness', 'BMI']:
    x = df[f].values
    p, p_p = pearsonr(x, y)
    s, s_p = spearmanr(x, y)
    k, k_p = kendalltau(x, y)
    rows.append({'Feature': f,
                 'Pearson r': p,
                 'p Pearson r': p_p,
                 'Spearman r': s, 
                 'p Spearman r': s_p,
                 'Kendall tau': k,
                 'p Kendall tau': k_p})
corr_df = pd.DataFrame(rows).round(2)
corr_df

,Feature,Pearson r,p Pearson r,Spearman r,p Spearman r,Kendall tau,p Kendall tau
0,SkinThickness,0.4400,0.0000,0.5400,0.0000,0.4200,0.0000
1,BMI,0.2000,0.0000,0.1900,0.0000,0.1400,0.0000
